#Regularisation: Ridge and Lasso Regression

**Introduction**

Imagine a data set which includes two input variables, X<sub>i</sub> and X<sub>j</sub>, X<sub>j</sub>=-X<sub>i</sub>, such that they have no genuine bearing on the output y. Still, a linear model that sets β<sub>j</sub>=-β<sub>i</sub> will have the same accuracy as the one from which X<sub>i</sub> and X<sub>j</sub> are removed, as  the two terms in the sum β<sub>i</sub>X<sub>i</sub>+β<sub>j</sub>X<sub>j</sub> would cancel out. This situation can be mitigated if the loss function of our linear regression, the ordinary least squares (OLS), is extended with an additional *penalty* term, which pushes down the parameters  β<sub>1</sub>... β<sub>p</sub>. Everything else being equal, this would allow us to reduce the parameters β<sub>i</sub> and β<sub>j</sub> of our example to as close to zero as possible without any loss of accuracy.

When the penalty term is proportional to L2 = Σ(β<sub>i</sub><sup>2</sup>), the resulting regression is known as [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html#sklearn.linear_model.Lasso) regression (with a loss function OLS+λ.L2), while using penalty term L1 = Σ|β<sub>i</sub>| (loss function OLS+λ.L1) corresponds to the so-called [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html#sklearn-linear-model-ridge) regression.

L1 and L2 are also known as *regularisation* terms. λ is referred to as the complexity parameter. You can also see the letter α (alpha) used instead of λ - this includes the entire scikit-learn documentation.

Using regularisation terms - L1, L2 or their combination (with the result known as [Elastic Net](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html?highlight=elastic%20net#sklearn-linear-model-elasticnet)) has as an effect a reduction of the variance of the model parameters β<sub>1</sub>... β<sub>p</sub> as we vary the training data. The price to pay is a larger model bias, i.e. the average (squared) error we end up with as we vary our training data.


**Exercise**

In this exercise you will use both Ridge and Lasso regression, and also plain Linear Regression to build regression models for the California house prices dataset. The task for this dataset is to learn a model that predicts the median price of a house (in a California district) from 8 variables describing that district.

To choose the correct complexity penalty for Ridge and Lasso regression you should use the built-in *cross-validation* (see online lecture) that is available in scikit-learn via the classes [RidgeCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html?highlight=ridgecv#sklearn.linear_model.RidgeCV) and [LassoCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html). Look up how to use them in the scikit-learn documentation.

To help you out here is the initial part of my Python program for doing this:


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import RidgeCV, LinearRegression, LassoCV
import numpy as np
california = fetch_california_housing()
print(california.data.shape)
print(california.target.shape)
print(california.feature_names)

(20640, 8)
(20640,)
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


**In your first experiment:**

*   learn from the first 15,000 datapoints
*   print out the learned parameter values for each predictor
*   compute the R<sup>2</sup> score on the remainder.

Do this for linear regression, ridge regression and lasso regression, then:

*   For ridge and lasso regression print out the complexity penalty value cross-validation found.
*   Inspect the learned parameter values for each predictor, and comment on their significance.


**In your second experiment:**

*  do the same except this time just train on the first 150 datapoints.

The results you get from the first and second experiments should be quite different - consider the reasons for that.



##Solution

(to the above two experiments combined)

In [ ]:
#Add your code here

myalphas = np.logspace(-6, 6, 13)
train_sizes = 15000, 150

models = LinearRegression, RidgeCV, LassoCV
model_names = 'linear regression', 'Ridge regression using CV', 'Lasso regression using CV'

for train_size in train_sizes:
    print('Training with this many datapoints: ', train_size)
    for i, model in enumerate(models):
        print('Using regression of type: ', model_names[i])
        if model == LinearRegression:
            reg = model()
        else:
            print('Trying these complexity parameter values (i.e.alphas): ', myalphas)
            reg = model(alphas=myalphas)
        reg.fit(california.data[:train_size],california.target[:train_size])

        if model != LinearRegression:
            print('Using cross-validation got this value for the complexity parameter: ', reg.alpha_)
        for i, name in enumerate(california.feature_names):
            print('Parameter for {0} is {1}'.format(name,reg.coef_[i]))

        print('Error on held out set is: ', reg.score(california.data[train_size:],california.target[train_size:]))
        print('\n\n')

Training with this many datapoints:  15000
Using regression of type:  linear regression
Parameter for MedInc is 0.4431337715103357
Parameter for HouseAge is 0.006888344329394301
Parameter for AveRooms is -0.10622562264813581
Parameter for AveBedrms is 0.621297542289487
Parameter for Population is -1.2245304356152398e-05
Parameter for AveOccup is -0.00781556745261362
Parameter for Latitude is -0.38535813241759675
Parameter for Longitude is -0.3678146164643453
Error on held out set is:  0.5931322421964103



Using regression of type:  Ridge regression using CV
Trying these complexity parameter values (i.e.alphas):  [1.e-06 1.e-05 1.e-04 1.e-03 1.e-02 1.e-01 1.e+00 1.e+01 1.e+02 1.e+03
 1.e+04 1.e+05 1.e+06]
Using cross-validation got this value for the complexity parameter:  10.0
Parameter for MedInc is 0.4418145201930521
Parameter for HouseAge is 0.006905176325659568
Parameter for AveRooms is -0.10379774538649356
Parameter for AveBedrms is 0.6087093269223149
Parameter for Population is 

##L1 and L2 regularisation with PyTorch

So, Lasso regression is simply a regression model that uses L1 regularisation;  a model that uses L2 regularisation is called Ridge regression. This means that we can easily modify our implementation of Linear regression by tweaking the loss function.

In [ ]:
import torch

# Function to compute the L1 loss of all weights in a model
def L1regloss(model):
  reg_loss = 0
  for param in model.parameters():
    reg_loss += param.abs().sum()
  return reg_loss

model = torch.nn.Linear(8,1)
X_train = torch.from_numpy(np.float32(california.data[:15000,:]))
y_train = torch.from_numpy(np.float32(california.target[:15000])).unsqueeze(1)
X_test = torch.from_numpy(np.float32(california.data[15000:,:]))
y_test = torch.from_numpy(np.float32(california.target[15000:])).unsqueeze(1)
epochs = 50

# Setup optimiser
optim = torch.optim.SGD(model.parameters(), lr=1e-7)
criterion = torch.nn.MSELoss()

# Main training loop
for epoch in range(epochs):
    y_predict = model(X_train)
    fit_loss = criterion(y_train,y_predict)
    # Use L1regloss for Lasso
    reg_loss = L1regloss(model)
    loss = fit_loss+reg_loss
    optim.zero_grad()
    loss.backward()
    optim.step()
    print('epoch {}, fit loss {}, reg loss {}'.format(epoch, fit_loss.item(), reg_loss.item()))

# Generalisation error
print(criterion(model(X_test),y_test))

epoch 0, fit loss 42459.77734375, reg loss 1.8606222867965698
epoch 1, fit loss 4739.3955078125, reg loss 1.783152461051941
epoch 2, fit loss 752.392578125, reg loss 1.7587603330612183
epoch 3, fit loss 330.4507141113281, reg loss 1.766374111175537
epoch 4, fit loss 285.2665100097656, reg loss 1.7687584161758423
epoch 5, fit loss 279.9020080566406, reg loss 1.7694425582885742
epoch 6, fit loss 278.747802734375, reg loss 1.769574522972107
epoch 7, fit loss 278.0400695800781, reg loss 1.769526720046997
epoch 8, fit loss 277.381103515625, reg loss 1.7694207429885864
epoch 9, fit loss 276.72882080078125, reg loss 1.7692959308624268
epoch 10, fit loss 276.07879638671875, reg loss 1.7691651582717896
epoch 11, fit loss 275.430419921875, reg loss 1.76903235912323
epoch 12, fit loss 274.78369140625, reg loss 1.7688992023468018
epoch 13, fit loss 274.1387023925781, reg loss 1.768766164779663
epoch 14, fit loss 273.49530029296875, reg loss 1.7686330080032349
epoch 15, fit loss 272.85357666015625,

**To do:**

Modify the loss function in the above solution to use the L2 loss as penalty instead of L1 loss, thus obtaining an implementation of Ridge regression.

In [ ]:
# Your code here - or edit the code above.

**Pytorch solution**:

In [ ]:
# Function to compute the L2 loss of all weights in a model
def L2regloss(model):
  reg_loss = 0
  for param in model.parameters():
    reg_loss += (param**2).sum()
  return reg_loss


model = torch.nn.Linear(8,1)
X_train = torch.from_numpy(np.float32(california.data[:15000,:]))
y_train = torch.from_numpy(np.float32(california.target[:15000])).unsqueeze(1)
X_test = torch.from_numpy(np.float32(california.data[15000:,:]))
y_test = torch.from_numpy(np.float32(california.target[15000:])).unsqueeze(1)
epochs = 50

# Setup optimiser
optim = torch.optim.SGD(model.parameters(), lr=1e-7)
criterion = torch.nn.MSELoss()

# Main training loop
for epoch in range(epochs):
    y_predict = model(X_train)
    fit_loss = criterion(y_train,y_predict)
    # Use L2regloss for Ridge and L1regloss for Lasso
    reg_loss = L2regloss(model)
    loss = fit_loss+reg_loss
    optim.zero_grad()
    loss.backward()
    optim.step()
    print('epoch {}, fit loss {}, reg loss {}'.format(epoch, fit_loss.item(), reg_loss.item()))

# Generalisation error
print(criterion(model(X_test),y_test))

epoch 0, fit loss 104520.3828125, reg loss 0.6084793210029602
epoch 1, fit loss 11278.044921875, reg loss 0.5805530548095703
epoch 2, fit loss 1423.3057861328125, reg loss 0.577472448348999
epoch 3, fit loss 381.2668151855469, reg loss 0.5770472288131714
epoch 4, fit loss 270.58526611328125, reg loss 0.5769126415252686
epoch 5, fit loss 258.3343505859375, reg loss 0.5768120884895325
epoch 6, fit loss 256.4873046875, reg loss 0.5767162442207336
epoch 7, fit loss 255.7412109375, reg loss 0.5766215324401855
epoch 8, fit loss 255.11288452148438, reg loss 0.5765272974967957
epoch 9, fit loss 254.49842834472656, reg loss 0.5764333605766296
epoch 10, fit loss 253.88681030273438, reg loss 0.5763395428657532
epoch 11, fit loss 253.2769317626953, reg loss 0.5762460827827454
epoch 12, fit loss 252.66864013671875, reg loss 0.5761527419090271
epoch 13, fit loss 252.06185913085938, reg loss 0.5760597586631775
epoch 14, fit loss 251.45668029785156, reg loss 0.575967013835907
epoch 15, fit loss 250.85